# Dedicated $\nu_\mu$CC radiative-correction simulation: truth and reco checks

`numucc_radcorr_1g_larger_v02_splines.root` (filetype `numucc_rad_corr_sim`) is a run-5 overlay sample made with
[uboone/ubsim PR #20](https://github.com/uboone/ubsim/pull/20) (`GENIEGenNGEM` + `AddRadCorrCollinearPhoton`). For every GENIE
CC event with an outgoing $\mu^\pm$, it adds one collinear QED photon sampled from the collinear function of
[Tomalak et al., PRD 106, 093006 (2022)](https://arxiv.org/abs/2204.11379), Eqs. (35)-(36):

$$
j(x,\eta) = \frac{\alpha}{\pi}\left[\frac{1}{2}\frac{1+x^2}{1-x}\ln(1+x^2\eta^2) - \frac{x}{1-x}\frac{x^2\eta^2}{1+x^2\eta^2}\right],
\qquad x = \frac{E_\mu}{E_\mathrm{tree}},\quad \eta = \frac{\Delta\theta\, E_\mathrm{tree}}{m_\mu},
$$

with $E_\mathrm{tree}$ the GENIE (tree-level) muon energy and $\Delta\theta$ the muon-photon opening angle; $\partial j/\partial\eta$ is the
density in $(x,\eta)$. The generator settings are:

* `RadCorrDeltaE: 0.020`: the photon needs $E_\gamma > 20$ MeV;
* `RadCorrMaxAngle: 60.`: the opening angle is at most $60^\circ$;
* the radiated muon is kept above its mass ($x > m_\mu/E_\mathrm{tree}$);
* `RadCorrForceEmission: false`: a photon is added with the physical probability $P(E_\mathrm{tree}) = \int_{m_\mu/E}^{1-\Delta E/E} j(x,\eta_\mathrm{max})\,dx$;
* `RadCorrRequireRadiation: true`: only radiating events are kept, but all POT is counted. The sample is therefore unweighted, and its rate per POT is the absolute radiative rate.

The kinematics are built as follows. The radiated muon and the photon share $E_\mathrm{tree}$ exactly. They sit on opposite sides of the tree-level muon
direction, with balanced transverse momentum and a uniform azimuth. The tree-level muon is kept in the GENIE record with status 3.

**Comparisons in this notebook**

1. **Sanity checks** on the sample: one status-3 muon with two status-1 daughters in every event; 4-momentum bookkeeping; tree alignment; the Wire-Cell truth photon is the generator photon.
2. **Paper figures.** Fig. 4 of the paper shows $d\sigma^\gamma/d\sigma_\mathrm{LO}$ for one photon above $\Delta E = 20$ MeV inside a cone
   $\Delta\theta$ (Eq. 36), for jet energies of 600 MeV, 2 GeV and 6 GeV. Fig. 5 shows its derivative in $\Delta\theta$. We reproduce the muon
   panels analytically and overlay the sample. The sample's ratio to LO uses the nominal run-5 $\nu$ overlay as the LO numuCC rate per POT.
3. **Absolute emission probability** $P(E_\mathrm{tree})$: sample rate / nominal numuCC rate, per POT.
4. **A Python re-implementation of the generator**, applied to every numuCC event of the nominal run-5 overlay and weighted by $P(E_\mathrm{tree})$.
   It predicts every true distribution of the sample. We compare it for:
   * the muon energy and angle;
   * the photon energy and angle;
   * the muon-photon invariant mass and opening angle.
5. **Reco**: the same variables from the Wire-Cell reco muon and primary shower (`T_PFeval`). We read them straight from the ROOT
   file, since the dataframes are being reprocessed. We show them at a few reco stages, alongside reco-vs-true maps and shower-finding efficiencies.

The generator-level comparisons (sections 2-4) are unweighted GENIE: the photon emission doesn't depend on the tune weight, so this is the
cleanest test. The reco section uses the CV weight (`weightTune * weightSpline`, sanitized like `genie_base_weight_expr`).

The nominal-overlay truth extraction takes ~1 min and is cached in `rad_corr_sim_cache/`. Delete the cache file to re-extract.

In [ ]:
import os, numpy as np, awkward as ak, uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from scipy.special import spence
mpl.rcParams['figure.dpi'] = 110

DATA_DIR = "/nevis/riverside/data/leehagaman/ngem/data_files"
SIM_FILE = f"{DATA_DIR}/numucc_radcorr_1g_larger_v02_splines.root"
NOM_FILE = f"{DATA_DIR}/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_5.root"
CACHE_DIR = "rad_corr_sim_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
PLOT_DIR = "../plots/numucc_rad_corr_sim"
os.makedirs(PLOT_DIR, exist_ok=True)

def save_and_show(fig, name):
    # every figure is saved as pdf and png in PLOT_DIR
    fig.savefig(f"{PLOT_DIR}/{name}.pdf", bbox_inches="tight")
    fig.savefig(f"{PLOT_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

# generator settings (prodgenie_bnb_nu_numucc_radcorr_1g_uboone.fcl), energies in GeV
ALPHA = 1.0 / 137.036
M_MU = 0.105658
DELTA_E = 0.020
MAX_ANGLE_DEG = 60.0

PER_POT = 1e21   # rates below are events per 1e21 POT

## 1. Theory functions

`j_collinear` and `dj_deta` are Eq. (35) and its $\eta$ derivative, written the same way as `RadCorrCollinearPhoton.h`. `eq36` is the
paper's closed form, which integrates $x$ from 0. `cone_fraction` integrates over the generator's region instead, which starts at $x = m_\mu/E_\mathrm{tree}$.
With $\Delta\theta = 60^\circ$, `cone_fraction` is the emission probability $P(E_\mathrm{tree})$.

In [ ]:
def j_collinear(x, eta):
    # Eq. (35)
    x, eta = np.asarray(x, float), np.asarray(eta, float)
    xe2 = x**2 * eta**2
    with np.errstate(divide='ignore', invalid='ignore'):
        r = ALPHA / np.pi * (0.5 * (1 + x**2) / (1 - x) * np.log1p(xe2) - x / (1 - x) * xe2 / (1 + xe2))
    return np.where((x > 0) & (x < 1) & (eta > 0), r, 0.0)


def dj_deta(x, eta):
    # d^2 sigma^gamma / (dx deta dsigma_LO)
    x, eta = np.asarray(x, float), np.asarray(eta, float)
    xe2 = x**2 * eta**2
    with np.errstate(divide='ignore', invalid='ignore'):
        r = ALPHA / np.pi * (x**2 * eta) / ((1 - x) * (1 + xe2)) * ((1 + x**2) - 2 * x / (1 + xe2))
    return np.where((x > 0) & (x < 1) & (eta > 0), r, 0.0)


def eq36(dtheta_deg, E_tree, dE=DELTA_E, m=M_MU):
    # paper Eq. (36): integral of j over x in [0, 1 - dE/E_tree]; Li2(z) = spence(1 - z)
    e = np.asarray(np.deg2rad(dtheta_deg) * E_tree / m, float)
    with np.errstate(divide='ignore', invalid='ignore'):
        L, A = np.log1p(e**2), np.arctan(e)
        r = ALPHA / np.pi * (0.5 * spence(1 + e**2) + 0.25 * L**2 + 0.25 * (2 / (1 + e**2) - 1 / e**2 - 3) * L + 9 / 4
                             + (L - e**2 / (1 + e**2)) * np.log(E_tree / dE) - A**2 - (1 + 1 / (1 + e**2)) * A / e)
    return np.where(e > 0, r, 0.0)


def cone_fraction(E_tree, dtheta_deg, dE=DELTA_E, m=M_MU, n=400):
    # Integral of j(x, eta(dtheta)) over the generator's region x in [m/E, 1 - dE/E]: the fraction of LO
    # events that radiate a photon with E_gamma > dE within dtheta of the muon. Midpoint rule in
    # v = ln(1 - x) = ln(E_gamma / E_tree), where the integrand j * (1 - x) is smooth. Broadcasts.
    E, th = np.broadcast_arrays(np.asarray(E_tree, float), np.asarray(dtheta_deg, float))
    v_lo = np.log(dE / E)[..., None]
    v_hi = np.log(np.clip(1 - m / E, 1e-300, None))[..., None]
    s = (np.arange(n) + 0.5) / n
    v = v_lo + (v_hi - v_lo) * s
    eta = (np.deg2rad(th) * E / m)[..., None]
    out = np.sum(j_collinear(1 - np.exp(v), eta) * np.exp(v), axis=-1) * (v_hi - v_lo)[..., 0] / n
    return np.where(v_hi[..., 0] > v_lo[..., 0], out, 0.0)


def P_emit(E_tree):
    return cone_fraction(E_tree, MAX_ANGLE_DEG)


# cross-check against the generator's own algorithm (linear midpoint sum with 1e5 steps)
def P_generator_style(E, n=100000):
    xmin, xmax = M_MU / E, 1 - DELTA_E / E
    x = xmin + (np.arange(n) + 0.5) * (xmax - xmin) / n
    return np.sum(j_collinear(x, np.deg2rad(MAX_ANGLE_DEG) * E / M_MU)) * (xmax - xmin) / n

for E in [0.15, 0.3, 0.6, 1.0, 2.0, 6.0]:
    print(f"E_tree = {E:4.2f} GeV: P = {P_emit(E):.5f} (generator algorithm {P_generator_style(E):.5f}),"
          f"  Eq. 36 at 60 deg (x from 0) = {eq36(60, E):.5f}")

## 2. Load the sample

The generator record comes from the gLEE `singlephotonana/vertex_tree` (`mctruth_daughters_*`). From it we take the tree-level muon (status 3)
and its two status-1 daughters, the radiated muon and the photon. Reco and selection information come from the Wire-Cell trees. The CV weight
comes from `nuselection`.

In [ ]:
def p4(arr_E, arr_px, arr_py, arr_pz, mask):
    # first masked particle of every event -> (N, 4) array of [E, px, py, pz]
    return np.column_stack([ak.to_numpy(ak.firsts(a[mask])) for a in (arr_E, arr_px, arr_py, arr_pz)]).astype(float)

f_sim = uproot.open(SIM_FILE)
sim_pot = f_sim["wcpselection/T_pot"].arrays(["pot_tor875good"], library="np")["pot_tor875good"].sum()

vt = f_sim["singlephotonana/vertex_tree"].arrays(
    ["run_number", "subrun_number", "event_number", "mctruth_num", "mctruth_cc_or_nc", "mctruth_lepton_pdg", "mctruth_lepton_E",
     "mctruth_daughters_pdg", "mctruth_daughters_status_code", "mctruth_daughters_trackID", "mctruth_daughters_mother_trackID",
     "mctruth_daughters_E", "mctruth_daughters_px", "mctruth_daughters_py", "mctruth_daughters_pz"])
pdg, st = vt["mctruth_daughters_pdg"], vt["mctruth_daughters_status_code"]
tid, mo = vt["mctruth_daughters_trackID"], vt["mctruth_daughters_mother_trackID"]
dE_, dpx, dpy, dpz = (vt[f"mctruth_daughters_{c}"] for c in ("E", "px", "py", "pz"))

is_tree = (abs(pdg) == 13) & (st == 3)
tree_tid = ak.firsts(tid[is_tree])
is_dau = mo == tree_tid
is_rmu = is_dau & (abs(pdg) == 13) & (st == 1)
is_gam = is_dau & (pdg == 22) & (st == 1)

sim = {
    "tree": p4(dE_, dpx, dpy, dpz, is_tree),
    "mu":   p4(dE_, dpx, dpy, dpz, is_rmu),
    "g":    p4(dE_, dpx, dpy, dpz, is_gam),
    "nu_pdg_sign": np.sign(ak.to_numpy(vt["mctruth_lepton_pdg"])),
}
n_sim = len(sim["tree"])
print(f"sample: {n_sim} events, {sim_pot:.4e} POT")
print("mctruth_num values:", dict(zip(*np.unique(ak.to_numpy(vt['mctruth_num']), return_counts=True))))
print("CC with |lepton pdg| = 13:", np.mean((ak.to_numpy(vt['mctruth_cc_or_nc']) == 0) & (abs(ak.to_numpy(vt['mctruth_lepton_pdg'])) == 13)))
print("lepton pdg (mu- / mu+):", dict(zip(*np.unique(ak.to_numpy(vt['mctruth_lepton_pdg']), return_counts=True))))
print("events with exactly one status-3 muon:", np.mean(ak.to_numpy(ak.sum(is_tree, axis=1)) == 1))
print("events with exactly one radiated muon and one photon daughter:",
      np.mean((ak.to_numpy(ak.sum(is_rmu, axis=1)) == 1) & (ak.to_numpy(ak.sum(is_gam, axis=1)) == 1)))
print("mctruth_lepton_E == tree-level muon E:", np.allclose(ak.to_numpy(vt["mctruth_lepton_E"]), sim["tree"][:, 0]))

resid = sim["tree"] - sim["mu"] - sim["g"]
p_tree = np.linalg.norm(sim["tree"][:, 1:], axis=1)
p_sum = np.linalg.norm(sim["mu"][:, 1:] + sim["g"][:, 1:], axis=1)
axis_ = sim["tree"][:, 1:] / p_tree[:, None]
perp = (sim["mu"][:, 1:] + sim["g"][:, 1:]) - np.sum((sim["mu"][:, 1:] + sim["g"][:, 1:]) * axis_, axis=1)[:, None] * axis_
print(f"\nenergy conservation: max |E_tree - E_mu - E_gamma| = {np.abs(resid[:, 0]).max() * 1e3:.2e} MeV")
print(f"(mu + gamma) momentum transverse to the tree-muon axis: max {np.linalg.norm(perp, axis=1).max() * 1e3:.2e} MeV")
print(f"|p_tree| - |p_mu + p_gamma| (expected >= 0: the mu-gamma system is heavier than m_mu): "
      f"median {np.median(p_tree - p_sum) * 1e3:.2f} MeV, 99% {np.percentile(p_tree - p_sum, 99) * 1e3:.1f} MeV, "
      f"min {np.min(p_tree - p_sum) * 1e3:.2e} MeV")

In [ ]:
# Wire-Cell / nuselection trees: alignment, truth cross-checks, reco and selection inputs
t_eval = f_sim["wcpselection/T_eval"].arrays(["run", "subrun", "event", "truth_vtxInside", "match_isFC"], library="np")
t_kine = f_sim["wcpselection/T_KINEvars"].arrays(["kine_reco_Enu"], library="np")
t_bdt = f_sim["wcpselection/T_BDTvars"].arrays(["numu_cc_flag", "numu_score"], library="np")
t_pf = f_sim["wcpselection/T_PFeval"].arrays(["run", "subrun", "event", "truth_pdg", "truth_mother", "truth_startMomentum",
                                              "truth_muonMomentum", "reco_muonMomentum", "reco_showerMomentum"])
t_ns = f_sim["nuselection/NeutrinoSelectionFilter"].arrays(["run", "sub", "evt", "weightTune", "weightSpline"], library="np")

rse_vt = np.column_stack([ak.to_numpy(vt[k]) for k in ("run_number", "subrun_number", "event_number")])
for name, rse in [("T_eval", np.column_stack([t_eval[k] for k in ("run", "subrun", "event")])),
                  ("T_PFeval", np.column_stack([ak.to_numpy(t_pf[k]) for k in ("run", "subrun", "event")])),
                  ("nuselection", np.column_stack([t_ns[k] for k in ("run", "sub", "evt")]))]:
    print(f"{name:12s} run/subrun/event aligned with vertex_tree: {np.mean(np.all(rse == rse_vt, axis=1)):.4f}")

# the Wire-Cell truth (Geant4 primaries) should contain the generator photon and radiated muon
prim_g = (t_pf["truth_pdg"] == 22) & (t_pf["truth_mother"] == 0)
n_prim_g = ak.to_numpy(ak.sum(prim_g, axis=1))
wc_g_E = ak.to_numpy(ak.max(t_pf["truth_startMomentum"][prim_g][:, :, 3], axis=1, mask_identity=False, initial=-1))
wc_mu = np.stack(ak.to_numpy(t_pf["truth_muonMomentum"]))
print("\nWire-Cell primary photons per event:", dict(zip(*np.unique(n_prim_g, return_counts=True))))
print(f"leading WC primary photon matches the generator photon energy (0.1 MeV): {np.mean(np.abs(wc_g_E - sim['g'][:, 0]) < 1e-4):.4f}")
print(f"WC truth_muonMomentum matches the radiated muon (0.1 MeV): {np.mean(np.abs(wc_mu[:, 3] - sim['mu'][:, 0]) < 1e-4):.4f}")

w = t_ns["weightTune"] * t_ns["weightSpline"]
cv_weight = np.where(np.isfinite(w) & (w > 0) & (w <= 30), w, 1.0)
print(f"\nCV weight: mean {cv_weight.mean():.3f}, fraction reset to 1: {np.mean(~(np.isfinite(w) & (w > 0) & (w <= 30))):.4f}")

reco = {
    "mu": np.stack(ak.to_numpy(t_pf["reco_muonMomentum"]))[:, [3, 0, 1, 2]].astype(float),   # -> [E, px, py, pz]
    "g":  np.stack(ak.to_numpy(t_pf["reco_showerMomentum"]))[:, [3, 0, 1, 2]].astype(float),
}
truth_in_fv = t_eval["truth_vtxInside"].astype(bool)

## 3. Nominal run-5 $\nu_\mu$CC (LO reference) and a Python copy of the generator

The LO reference is every event of the nominal run-5 $\nu$ overlay with a CC $\mu^\pm$ (`mctruth_cc_or_nc == 0`, $|$lepton pdg$|$ = 13).
This is the same population the generator acts on, over the whole cryostat. For each event we keep the GENIE muon 4-momentum. The sample was
generated with the same GENIE configuration, so

$$ \frac{N_\mathrm{sim}(\text{bin})/\mathrm{POT}_\mathrm{sim}}{N_\mathrm{nom}(\text{bin})/\mathrm{POT}_\mathrm{nom}} \;\to\; \frac{d\sigma^\gamma}{d\sigma_\mathrm{LO}}. $$

The Python generator copy then radiates a photon from **every** nominal numuCC muon, with weight $P(E_\mathrm{tree})/\mathrm{POT}_\mathrm{nom}$. It samples
$(x, \eta)$ from $\partial j/\partial\eta$ by exact inverse-CDF sampling (not the generator's rejection sampling), then builds the kinematics exactly as
`AddRadCorrCollinearPhoton.h` does. The result is an independent, high-statistics prediction of every true distribution in the sample.

In [ ]:
def extract_nominal_numucc(path, step_size=100000):
    t = uproot.open(path)["singlephotonana/vertex_tree"]
    out = {k: [] for k in ("E", "px", "py", "pz", "lep_pdg")}
    n_tot = 0
    for a in t.iterate(["mctruth_cc_or_nc", "mctruth_lepton_pdg", "mctruth_lepton_E", "mctruth_daughters_pdg",
                        "mctruth_daughters_status_code", "mctruth_daughters_E", "mctruth_daughters_px",
                        "mctruth_daughters_py", "mctruth_daughters_pz"], step_size=step_size):
        n_tot += len(a)
        a = a[(a["mctruth_cc_or_nc"] == 0) & (abs(a["mctruth_lepton_pdg"]) == 13)]
        m = (abs(a["mctruth_daughters_pdg"]) == 13) & (a["mctruth_daughters_status_code"] == 1)
        # the primary lepton: the status-1 muon whose energy matches mctruth_lepton_E
        k = ak.argmin(abs(a["mctruth_daughters_E"][m] - a["mctruth_lepton_E"]), axis=1, keepdims=True)
        for c in ("E", "px", "py", "pz"):
            out[c].append(ak.to_numpy(ak.flatten(a[f"mctruth_daughters_{c}"][m][k])))
        out["lep_pdg"].append(ak.to_numpy(a["mctruth_lepton_pdg"]))
        print(f"  {n_tot} events scanned", end="\r")
    print()
    res = {k: np.concatenate(v) for k, v in out.items()}
    res["n_events_total"] = n_tot
    res["pot"] = uproot.open(path)["wcpselection/T_pot"].arrays(["pot_tor875good"], library="np")["pot_tor875good"].sum()
    return res

nom_cache = f"{CACHE_DIR}/nominal_run5_numucc_muons.npz"
if not os.path.exists(nom_cache):
    np.savez(nom_cache, **extract_nominal_numucc(NOM_FILE))
nom_npz = np.load(nom_cache)
nom = {"tree": np.column_stack([nom_npz[c] for c in ("E", "px", "py", "pz")]).astype(float)}
nom_pot = float(nom_npz["pot"])
print(f"nominal run-5 nu overlay: {int(nom_npz['n_events_total'])} events, {len(nom['tree'])} CC mu+- events, {nom_pot:.4e} POT")
mass = np.sqrt(np.clip(nom['tree'][:, 0]**2 - np.sum(nom['tree'][:, 1:]**2, axis=1), 0, None))
print(f"  selected lepton is a muon: fraction with |m - m_mu| < 1 MeV = {np.mean(np.abs(mass - M_MU) < 1e-3):.4f} "
      f"(1 MeV tolerance for float32 rounding)")

In [ ]:
def sample_x_eta(E, rng, n_grid=800, chunk=5000):
    # Sample (x, eta) from dj/deta over the generator region, one pair per E_tree.
    #   x: inverse CDF of the marginal j(x, eta_max), on a midpoint grid in v = ln(1 - x)
    #   eta | x: solve j(x, eta) = r * j(x, eta_max) by bisection (j increases with eta)
    E = np.asarray(E, float)
    x_out, eta_out = np.empty_like(E), np.empty_like(E)
    s = (np.arange(n_grid) + 0.5) / n_grid
    for i0 in range(0, len(E), chunk):
        Ec = E[i0:i0 + chunk][:, None]
        v_lo, v_hi = np.log(DELTA_E / Ec), np.log(1 - M_MU / Ec)
        eta_max = np.deg2rad(MAX_ANGLE_DEG) * Ec / M_MU
        dens = j_collinear(1 - np.exp(v_lo + (v_hi - v_lo) * s), eta_max) * np.exp(v_lo + (v_hi - v_lo) * s)
        cdf = np.cumsum(dens, axis=1)
        cdf /= cdf[:, -1:]
        r = rng.random(len(Ec))[:, None]
        k = np.minimum((cdf < r).sum(axis=1), n_grid - 1)
        c_prev = np.where(k > 0, np.take_along_axis(cdf, np.maximum(k - 1, 0)[:, None], axis=1)[:, 0], 0.0)
        c_here = np.take_along_axis(cdf, k[:, None], axis=1)[:, 0]
        frac = (r[:, 0] - c_prev) / np.maximum(c_here - c_prev, 1e-300)
        v = v_lo[:, 0] + (v_hi - v_lo)[:, 0] * (k + frac) / n_grid
        x = 1 - np.exp(v)

        target = rng.random(len(x)) * j_collinear(x, eta_max[:, 0])
        lo, hi = np.zeros_like(x), eta_max[:, 0].copy()
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            below = j_collinear(x, mid) < target
            lo, hi = np.where(below, mid, lo), np.where(below, hi, mid)
        x_out[i0:i0 + chunk], eta_out[i0:i0 + chunk] = x, 0.5 * (lo + hi)
    return x_out, eta_out


def radiate(tree, x, eta, rng):
    # same construction as AddRadCorrCollinearPhoton.h: returns ([E,p] radiated muon, [E,p] photon)
    E_tree, p3 = tree[:, 0], tree[:, 1:]
    theta = eta * M_MU / E_tree
    E_l = x * E_tree
    p_l = np.sqrt(np.clip(E_l**2 - M_MU**2, 0, None))
    E_g = (1 - x) * E_tree
    a = np.arctan2(E_g * np.sin(theta), p_l + E_g * np.cos(theta))
    b = theta - a
    ax = p3 / np.linalg.norm(p3, axis=1)[:, None]
    helper = np.where(np.abs(ax[:, 2:3]) < 0.9, [[0, 0, 1]], [[1, 0, 0]])
    e1 = np.cross(ax, helper); e1 /= np.linalg.norm(e1, axis=1)[:, None]
    e2 = np.cross(ax, e1)
    phi = rng.uniform(0, 2 * np.pi, len(x))[:, None]
    tr = np.cos(phi) * e1 + np.sin(phi) * e2
    l_dir = np.cos(a)[:, None] * ax + np.sin(a)[:, None] * tr
    g_dir = np.cos(b)[:, None] * ax - np.sin(b)[:, None] * tr
    return (np.column_stack([E_l, p_l[:, None] * l_dir]), np.column_stack([E_g, E_g[:, None] * g_dir]))


rng = np.random.default_rng(20261007)
eligible = nom["tree"][:, 0] > M_MU + DELTA_E
toy = {"tree": nom["tree"][eligible]}
x_toy, eta_toy = sample_x_eta(toy["tree"][:, 0], rng)
toy["mu"], toy["g"] = radiate(toy["tree"], x_toy, eta_toy, rng)
toy_w = P_emit(toy["tree"][:, 0]) / nom_pot * PER_POT        # events per 1e21 POT
sim_w = np.full(n_sim, PER_POT / sim_pot)                    # unweighted sample, events per 1e21 POT
print(f"generator copy: {len(x_toy)} radiated nominal muons, predicted rate {toy_w.sum():.1f} per 1e21 POT; "
      f"sample: {sim_w.sum():.1f} +- {np.sqrt(n_sim) * PER_POT / sim_pot:.1f} per 1e21 POT  "
      f"(ratio {sim_w.sum() / toy_w.sum():.3f} +- {np.sqrt(n_sim) / n_sim * sim_w.sum() / toy_w.sum():.3f})")

In [ ]:
def kin(mu, g):
    # derived variables: energies in GeV, angles w.r.t. the beam (z), opening angle in deg, mass in GeV
    p_mu, p_g = np.linalg.norm(mu[:, 1:], axis=1), np.linalg.norm(g[:, 1:], axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        cos_open = np.sum(mu[:, 1:] * g[:, 1:], axis=1) / (p_mu * p_g)
        tot = mu + g
        m2 = tot[:, 0]**2 - np.sum(tot[:, 1:]**2, axis=1)
        return {
            "E_mu": mu[:, 0], "cos_mu": mu[:, 3] / p_mu,
            "E_g": g[:, 0], "cos_g": g[:, 3] / p_g,
            "open": np.degrees(np.arccos(np.clip(cos_open, -1, 1))),
            "M": np.sqrt(np.clip(m2, 0, None)),
            "E_sum": mu[:, 0] + g[:, 0],
        }

sim_k = kin(sim["mu"], sim["g"])
toy_k = kin(toy["mu"], toy["g"])
sim_k["E_tree"], toy_k["E_tree"] = sim["tree"][:, 0], toy["tree"][:, 0]
sim_k["x"] = sim["mu"][:, 0] / sim["tree"][:, 0]
sim_k["eta"] = np.radians(sim_k["open"]) * sim["tree"][:, 0] / M_MU
print(f"sample: E_gamma min {sim_k['E_g'].min() * 1e3:.2f} MeV, opening angle max {sim_k['open'].max():.2f} deg, "
      f"x min - m/E_tree: {np.min(sim_k['x'] - M_MU / sim_k['E_tree']):.2e}")

## 4. Paper Figs. 4 and 5: photon inside a cone of half-angle $\Delta\theta$

The left panels reproduce the muon panels of the paper's Figs. 4 and 5 at jet energies 600 MeV, 2 GeV and 6 GeV:
* solid: Eq. (36);
* dashed: the generator's region, where the radiated muon stays above $m_\mu$.

The right panels show the sample in $E_\mathrm{tree}$ slices around 600 MeV and 2 GeV. Above 5 GeV the sample has almost no events.
Each slice is drawn as cumulative events with opening angle $<\Delta\theta$, divided by LO. The theory there is `cone_fraction` averaged over the nominal events in the slice.

In [ ]:
th = np.linspace(0.05, 60, 600)
slices = [(0.5, 0.7, "C0"), (1.6, 2.4, "C1")]
E_nom = nom["tree"][:, 0]

fig, axs = plt.subplots(2, 2, figsize=(12, 8.5))
for E, c in [(0.6, "C0"), (2.0, "C1"), (6.0, "C2")]:
    axs[0, 0].plot(th, eq36(th, E), c=c, label=f"Eq. 36, $E$ = {E:g} GeV")
    axs[0, 0].plot(th, cone_fraction(E, th), c=c, ls="--")
    d = np.gradient(eq36(th, E), th)
    axs[1, 0].plot(th, d, c=c, label=f"$E$ = {E:g} GeV")
    axs[1, 0].plot(th, np.gradient(cone_fraction(E, th), th), c=c, ls="--")
axs[0, 0].plot([], [], c="k", ls="--", label=r"generator region ($x > m_\mu/E$)")
axs[0, 0].set(xlabel=r"$\Delta\theta$ (deg)", ylabel=r"$d\sigma^\gamma / d\sigma_\mathrm{LO}$", title="Fig. 4 (muon), $\\Delta E$ = 20 MeV")
axs[1, 0].set(xlabel=r"$\Delta\theta$ (deg)", ylabel=r"$\frac{d}{d\Delta\theta}\, d\sigma^\gamma / d\sigma_\mathrm{LO}$ (1/deg)",
              title="Fig. 5 (muon)", yscale="log", ylim=(1e-5, None))
axs[0, 0].legend(fontsize=8); axs[1, 0].legend(fontsize=8)

pts = np.array([1, 2, 3, 5, 7.5, 10, 15, 20, 25, 30, 40, 50, 60])
edges = np.linspace(0, 60, 21)
for lo, hi, c in slices:
    in_nom = (E_nom > lo) & (E_nom < hi)
    in_sim = (sim_k["E_tree"] > lo) & (sim_k["E_tree"] < hi)
    lo_rate = in_nom.sum() / nom_pot                       # LO events per POT in the slice
    E_s = E_nom[in_nom][::max(1, in_nom.sum() // 3000)]    # subsample for the theory average
    theo = np.array([cone_fraction(E_s, t).mean() for t in th])
    n_cum = np.array([np.sum(in_sim & (sim_k["open"] < t)) for t in pts])
    meas = n_cum / sim_pot / lo_rate
    axs[0, 1].errorbar(pts, meas, yerr=np.sqrt(n_cum) / sim_pot / lo_rate, fmt="o", c=c, ms=4,
                       label=f"sample, {lo:g} < $E_\\mathrm{{tree}}$ < {hi:g} GeV ({in_sim.sum()} events)")
    axs[0, 1].plot(th, theo, c=c, label="theory (generator region), averaged over the slice")
    n_bin, _ = np.histogram(sim_k["open"][in_sim], edges)
    width = np.diff(edges)
    axs[1, 1].errorbar(0.5 * (edges[1:] + edges[:-1]), n_bin / sim_pot / lo_rate / width,
                       yerr=np.sqrt(n_bin) / sim_pot / lo_rate / width, fmt="o", c=c, ms=4)
    axs[1, 1].plot(th, np.gradient(theo, th), c=c)
axs[0, 1].set(xlabel=r"$\Delta\theta$ (deg)", ylabel=r"$N(\theta_{\mu\gamma}<\Delta\theta)$ / $N_\mathrm{LO}$ (per POT)",
              title="sample vs Fig. 4 theory")
axs[1, 1].set(xlabel=r"$\Delta\theta$ (deg)", ylabel=r"$dN/d\theta_{\mu\gamma}$ / $N_\mathrm{LO}$ (1/deg)",
              title="sample vs Fig. 5 theory", yscale="log", ylim=(1e-5, None))
axs[0, 1].legend(fontsize=8)
for a in axs.flat:
    a.grid(alpha=0.3)
plt.tight_layout(); save_and_show(fig, "paper_figs_4_5_comparison")

## 5. Absolute emission probability $P(E_\mathrm{tree})$

This is the sample's rate per POT divided by the nominal LO numuCC rate per POT, in bins of the tree-level muon energy. We compare it with the
generator's $P(E_\mathrm{tree})$, both the curve and its average over the nominal events in each bin. The Eq. (36) value at $60^\circ$ is also shown; it
integrates $x$ from 0.

The nominal overlay's statistical error is negligible. The sample's error is Poisson.

In [ ]:
eb = np.array([0.125, 0.2, 0.3, 0.4, 0.5, 0.6, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0])
n_s, _ = np.histogram(sim_k["E_tree"], eb)
n_n, _ = np.histogram(E_nom, eb)
p_bin = np.array([P_emit(E_nom[(E_nom > a) & (E_nom < b)]).mean() for a, b in zip(eb[:-1], eb[1:])])
P_meas = (n_s / sim_pot) / (n_n / nom_pot)
P_err = np.sqrt(n_s) / sim_pot / (n_n / nom_pot)

Eg = np.geomspace(0.13, 6, 300)
fig, (a0, a1) = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
a0.plot(Eg, P_emit(Eg), "k", label=r"$P(E_\mathrm{tree})$ (generator region, $\Delta\theta < 60^\circ$)")
a0.plot(Eg, eq36(60, Eg), "k:", label=r"Eq. 36 at $60^\circ$ ($x$ from 0)")
a0.stairs(p_bin, eb, color="C1", label="$P$ averaged over nominal events in bin")
xc = 0.5 * (eb[1:] + eb[:-1])
a0.errorbar(xc, P_meas, yerr=P_err, fmt="o", c="C0", ms=4, label="sample / nominal LO (per POT)")
a0.set(ylabel="emission probability", xscale="log"); a0.legend(fontsize=8); a0.grid(alpha=0.3)
a1.errorbar(xc, P_meas / p_bin, yerr=P_err / p_bin, fmt="o", c="C0", ms=4)
a1.axhline(1, c="k", lw=0.8)
a1.set(xlabel=r"$E_\mathrm{tree}$ = GENIE muon energy (GeV)", ylabel="sample / theory", ylim=(0.7, 1.3)); a1.grid(alpha=0.3)
plt.tight_layout(); save_and_show(fig, "emission_probability_vs_E_tree")

print(" E_tree bin [GeV]   N_sample   P_measured        P_theory   ratio")
for i in range(len(xc)):
    print(f"  {eb[i]:5.3f}-{eb[i+1]:<5.3f}   {n_s[i]:7d}   {P_meas[i]:.5f}+-{P_err[i]:.5f}   {p_bin[i]:.5f}   {P_meas[i] / p_bin[i]:.3f}")
tot = n_s.sum() / sim_pot / (toy_w.sum() / PER_POT)
print(f"total sample rate / generator-copy rate = {tot:.3f} +- {tot / np.sqrt(n_s.sum()):.3f}")

## 6. Generator variables $(x, \eta)$ and $(E_\gamma, \theta_{\mu\gamma})$

These are the sample against the Python generator copy, both in events per $10^{21}$ POT. The copy's statistical error is small: about 450k nominal numuCC events,
each radiated once with weight $P$.

In [ ]:
def ratio_plot(var_s, var_t, bins, xlabel, ax, axr, logx=False, logy=False, sw=None, tw=None, label_s="sample", label_t="generator copy (nominal run 5)"):
    sw = sim_w if sw is None else sw
    tw = toy_w if tw is None else tw
    hs, _ = np.histogram(var_s, bins, weights=sw)
    es = np.sqrt(np.histogram(var_s, bins, weights=sw**2)[0])
    ht, _ = np.histogram(var_t, bins, weights=tw)
    et = np.sqrt(np.histogram(var_t, bins, weights=tw**2)[0])
    xc = np.sqrt(bins[1:] * bins[:-1]) if logx else 0.5 * (bins[1:] + bins[:-1])
    ax.stairs(ht, bins, color="C1", label=label_t)
    ax.errorbar(xc, hs, yerr=es, fmt="o", c="k", ms=3, label=label_s)
    with np.errstate(invalid="ignore", divide="ignore"):
        axr.errorbar(xc, hs / ht, yerr=es / ht, fmt="o", c="k", ms=3)
        axr.fill_between(bins, np.append(1 - et / ht, 1), np.append(1 + et / ht, 1), step="post", color="C1", alpha=0.3)
    axr.axhline(1, c="k", lw=0.8); axr.set_ylim(0.5, 1.5)
    axr.set_xlabel(xlabel); ax.set_ylabel("events / 1e21 POT"); axr.set_ylabel("sample / copy")
    if logx: ax.set_xscale("log"); axr.set_xscale("log")
    if logy: ax.set_yscale("log")
    ax.grid(alpha=0.3); axr.grid(alpha=0.3)
    chi2 = np.nansum(((hs - ht) / np.sqrt(es**2 + et**2))[hs + ht > 0]**2)
    ax.set_title(f"$\\chi^2$/nbins = {chi2:.1f}/{np.sum(hs + ht > 0)}", fontsize=9)

def grid_of_ratio_plots(n):
    fig = plt.figure(figsize=(5 * n, 5))
    gs = fig.add_gridspec(2, n, height_ratios=[3, 1.2], hspace=0.05)
    return fig, [(fig.add_subplot(gs[0, i]), fig.add_subplot(gs[1, i])) for i in range(n)]

toy_k["x"] = x_toy
toy_k["eta"] = eta_toy
fig, pairs = grid_of_ratio_plots(3)
ratio_plot(1 - sim_k["x"], 1 - toy_k["x"], np.geomspace(3e-3, 1, 31), r"$1-x = E_\gamma/E_\mathrm{tree}$", *pairs[0], logx=True)
ratio_plot(sim_k["eta"], toy_k["eta"], np.geomspace(0.05, 100, 31), r"$\eta = \theta_{\mu\gamma} E_\mathrm{tree}/m_\mu$", *pairs[1], logx=True)
ratio_plot(sim_k["E_tree"], toy_k["E_tree"], np.linspace(0.1, 3, 30), r"$E_\mathrm{tree}$ (GeV)", *pairs[2])
for a, _ in pairs: a.legend(fontsize=8); a.tick_params(labelbottom=False)
save_and_show(fig, "generator_variables_1d")

In [ ]:
def pull_2d(xs, ys, xt, yt, bx, by, xlabel, ylabel, name, hlines=()):
    # Sample, generator copy, and pull (sample - copy) / sigma, on the same binning. Geometric bins get log axes.
    Hs, _, _ = np.histogram2d(xs, ys, [bx, by], weights=sim_w)
    Ht, _, _ = np.histogram2d(xt, yt, [bx, by], weights=toy_w)
    Ht_err2, _, _ = np.histogram2d(xt, yt, [bx, by], weights=toy_w**2)
    Ns, _, _ = np.histogram2d(xs, ys, [bx, by])
    pred_n = Ht / sim_w[0]                       # expected sample counts
    with np.errstate(invalid="ignore", divide="ignore"):
        pull = np.where(pred_n > 0, (Ns - pred_n) / np.sqrt(pred_n + Ht_err2 / sim_w[0]**2), np.nan)

    fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
    vmax = max(Hs.max(), Ht.max())
    for a, H, t in [(axs[0], Hs, "sample"), (axs[1], Ht, "generator copy")]:
        m = a.pcolormesh(bx, by, np.where(H > 0, H, np.nan).T, norm=LogNorm(vmin=vmax * 1e-3, vmax=vmax), cmap="viridis")
        plt.colorbar(m, ax=a, label="events / 1e21 POT"); a.set_title(t)
    m = axs[2].pcolormesh(bx, by, pull.T, cmap="RdBu_r", vmin=-4, vmax=4)
    plt.colorbar(m, ax=axs[2], label="(sample - copy) / sigma")
    axs[2].set_title(f"pull: mean {np.nanmean(pull[pred_n > 5]):.2f}, rms {np.nanstd(pull[pred_n > 5]):.2f} (bins with > 5 expected)")
    for a in axs:
        a.set(xlabel=xlabel, ylabel=ylabel)
        if is_geom(bx): a.set_xscale("log")
        if is_geom(by): a.set_yscale("log")
        for y, ls, lab in hlines:
            a.axhline(y, c="k", ls=ls, lw=1, label=lab)
    if hlines: axs[0].legend(fontsize=8, loc="lower left")
    plt.tight_layout(); save_and_show(fig, name)
    return pull, pred_n

def is_geom(b):
    return b[0] > 0 and np.allclose(np.diff(np.log(b)), np.log(b[1] / b[0]))

# (E_gamma, opening angle)
pull_2d(sim_k["E_g"], sim_k["open"], toy_k["E_g"], toy_k["open"], np.geomspace(0.02, 2.0, 21), np.linspace(0, 60, 21),
        r"true $E_\gamma$ (GeV)", r"true $\theta_{\mu\gamma}$ (deg)", "true_Eg_vs_opening_angle_2d")

The same comparison in the generator's own variables, $(1-x, \eta)$, on log axes. The edges depend on $E_\mathrm{tree}$:
$1-x > \Delta E/E_\mathrm{tree}$ sets the lower edge in $1-x$, $x > m_\mu/E_\mathrm{tree}$ the upper edge, and $\eta < \eta_\mathrm{max} = (60^\circ) E_\mathrm{tree}/m_\mu$ the upper edge in
$\eta$. The dashed line is $\eta = 1$, where the opening angle equals $m_\mu/E_\mathrm{tree}$. The dotted lines are $\theta_{\mu\gamma} = 10^\circ$ and $30^\circ$
at the sample's median $E_\mathrm{tree}$. Above roughly $10^\circ$, $(\Delta\theta)^2$ is no longer small and Eq. 35 is being extrapolated.

In [ ]:
E_med = float(np.ma.median(sim_k["E_tree"]))
pull_x_eta, _ = pull_2d(1 - sim_k["x"], sim_k["eta"], 1 - toy_k["x"], toy_k["eta"], np.geomspace(3e-3, 1, 21), np.geomspace(0.05, 100, 21),
                        r"$1-x = E_\gamma/E_\mathrm{tree}$", r"$\eta = \theta_{\mu\gamma} E_\mathrm{tree}/m_\mu$", "true_x_eta_2d",
                        hlines=[(1, "--", r"$\eta = 1$"),
                                (np.radians(10) * E_med / M_MU, ":", rf"$10^\circ$ at $E_\mathrm{{tree}}$ = {E_med:.2f} GeV"),
                                (np.radians(30) * E_med / M_MU, ":", rf"$30^\circ$ at $E_\mathrm{{tree}}$ = {E_med:.2f} GeV")])
print(f"median E_tree = {E_med:.2f} GeV: theta = 10 deg -> eta = {np.radians(10) * E_med / M_MU:.1f}, "
      f"30 deg -> {np.radians(30) * E_med / M_MU:.1f}, 60 deg -> {np.radians(60) * E_med / M_MU:.1f}")

## 7. True kinematics: muon, photon, invariant mass, opening angle

The black points are the sample. The orange line is the Python generator copy run on the nominal run-5 numuCC muons. On the muon panels, the dotted line is the
tree-level (GENIE) muon of the sample's events, which shows the shift from radiation. Angles are measured relative to the beam ($z$). Rates are unweighted, all vertices,
per $10^{21}$ POT.

In [ ]:
fig, pairs = grid_of_ratio_plots(3)
ratio_plot(sim_k["E_mu"], toy_k["E_mu"], np.linspace(0.1, 2.5, 25), r"true $E_\mu$ (GeV)", *pairs[0])
pairs[0][0].stairs(np.histogram(sim["tree"][:, 0], np.linspace(0.1, 2.5, 25), weights=sim_w)[0], np.linspace(0.1, 2.5, 25),
                   color="k", ls=":", label="sample, tree-level muon")
ratio_plot(sim_k["cos_mu"], toy_k["cos_mu"], np.linspace(-1, 1, 21), r"true $\cos\theta_\mu$", *pairs[1])
tree_cos = sim["tree"][:, 3] / np.linalg.norm(sim["tree"][:, 1:], axis=1)
pairs[1][0].stairs(np.histogram(tree_cos, np.linspace(-1, 1, 21), weights=sim_w)[0], np.linspace(-1, 1, 21),
                   color="k", ls=":", label="sample, tree-level muon")
ratio_plot(sim_k["M"] * 1e3, toy_k["M"] * 1e3, np.linspace(100, 500, 41), r"true $M_{\mu\gamma}$ (MeV)", *pairs[2], logy=True)
for a, _ in pairs: a.legend(fontsize=8); a.tick_params(labelbottom=False)
save_and_show(fig, "true_muon_kinematics")

fig, pairs = grid_of_ratio_plots(3)
ratio_plot(sim_k["E_g"] * 1e3, toy_k["E_g"] * 1e3, np.geomspace(20, 2000, 26), r"true $E_\gamma$ (MeV)", *pairs[0], logx=True, logy=True)
ratio_plot(sim_k["cos_g"], toy_k["cos_g"], np.linspace(-1, 1, 21), r"true $\cos\theta_\gamma$", *pairs[1])
ratio_plot(sim_k["open"], toy_k["open"], np.linspace(0, 60, 31), r"true $\theta_{\mu\gamma}$ (deg)", *pairs[2], logy=True)
for a, _ in pairs: a.legend(fontsize=8); a.tick_params(labelbottom=False)
save_and_show(fig, "true_photon_kinematics")

In [ ]:
def hist2d_pair(var_pairs, bins_pairs, labels, weights_pairs, titles, name, suptitle=None, log=True, shared_scale=True):
    fig, axs = plt.subplots(1, len(var_pairs), figsize=(5.6 * len(var_pairs), 4.4))
    Hs_ = [np.histogram2d(vx, vy, [bx_, by_], weights=w_)[0] for (vx, vy), (bx_, by_), w_ in zip(var_pairs, bins_pairs, weights_pairs)]
    for a, H, (bx_, by_), (lx, ly), t in zip(np.atleast_1d(axs), Hs_, bins_pairs, labels, titles):
        vmax = max(h.max() for h in Hs_) if shared_scale else H.max()
        norm = LogNorm(vmin=vmax * 1e-3, vmax=vmax) if log else mpl.colors.Normalize(0, vmax)
        m = a.pcolormesh(bx_, by_, np.where(H > 0, H, np.nan).T if log else H.T, norm=norm, cmap="viridis")
        plt.colorbar(m, ax=a, label="events / 1e21 POT")
        a.set(xlabel=lx, ylabel=ly, title=t)
        if is_geom(bx_): a.set_xscale("log")
        if is_geom(by_): a.set_yscale("log")
    if suptitle: fig.suptitle(suptitle)
    plt.tight_layout(); save_and_show(fig, name)

bE_mu, bcos = np.linspace(0.1, 2.5, 25), np.linspace(-1, 1, 21)
bE_g = np.geomspace(0.02, 2.0, 21)
lab_mu = (r"true $E_\mu$ (GeV)", r"true $\cos\theta_\mu$")
lab_g = (r"true $E_\gamma$ (GeV)", r"true $\cos\theta_\gamma$")
hist2d_pair([(sim_k["E_mu"], sim_k["cos_mu"]), (toy_k["E_mu"], toy_k["cos_mu"])], [(bE_mu, bcos)] * 2, [lab_mu] * 2,
            [sim_w, toy_w], ["sample", "generator copy"], "true_muon_E_vs_cos_2d", "true muon energy vs angle")
hist2d_pair([(sim_k["E_g"], sim_k["cos_g"]), (toy_k["E_g"], toy_k["cos_g"])], [(bE_g, bcos)] * 2, [lab_g] * 2,
            [sim_w, toy_w], ["sample", "generator copy"], "true_photon_E_vs_cos_2d", "true photon energy vs angle")
hist2d_pair([(sim_k["E_mu"], sim_k["E_g"]), (sim_k["E_g"], sim_k["open"])],
            [(bE_mu, bE_g), (bE_g, np.linspace(0, 60, 21))],
            [(r"true $E_\mu$ (GeV)", r"true $E_\gamma$ (GeV)"), (r"true $E_\gamma$ (GeV)", r"true $\theta_{\mu\gamma}$ (deg)")],
            [sim_w, sim_w], ["sample", "sample"], "true_muon_photon_correlations_2d", "true muon-photon correlations", shared_scale=False)

## 8. Reco

We use the Wire-Cell reco muon (`reco_muonMomentum`, a 4-vector in GeV with $-1$ sentinels) and the primary shower (`reco_showerMomentum`), treated as a massless
photon candidate. From these we build the reco versions of the same variables. The stages are:

* **presel**: Wire-Cell generic neutrino selection, `kine_reco_Enu > 0`;
* **presel + reco $\mu$ + reco shower**: a reco muon and a primary shower both exist;
* **numuCC + reco $\mu$ + reco shower**: the Wire-Cell numuCC selection (`numu_cc_flag >= 0 && numu_score > 0.9`), plus a reco muon and a primary shower.

The BDT selections need the dataframe variables, so they are not applied here. Rates are CV-weighted, per $10^{21}$ POT.

In [ ]:
cvw = cv_weight * PER_POT / sim_pot
has_mu = reco["mu"][:, 0] > 0
has_sh = reco["g"][:, 0] > 0
presel = t_kine["kine_reco_Enu"] > 0
numucc = (t_bdt["numu_cc_flag"] >= 0) & (t_bdt["numu_score"] > 0.9)
fc = t_eval["match_isFC"].astype(bool)
stages = {
    "all": np.ones(n_sim, bool),
    "presel": presel,
    "presel + reco mu": presel & has_mu,
    "presel + reco shower": presel & has_sh,
    "presel + reco mu + reco shower": presel & has_mu & has_sh,
    "numuCC": numucc,
    "numuCC + reco shower": numucc & has_sh,
    "numuCC + reco mu + reco shower": numucc & has_mu & has_sh,
    "numuCC + reco shower + FC": numucc & has_sh & fc,
}
print(f"{'stage':34s} {'N':>6s} {'rate/1e21':>10s} {'frac':>6s} {'in-FV frac':>10s}")
for name, m in stages.items():
    print(f"{name:34s} {m.sum():6d} {cvw[m].sum():10.1f} {cvw[m].sum() / cvw.sum():6.3f} {cvw[m & truth_in_fv].sum() / max(cvw[m].sum(), 1e-9):10.3f}")

# -1 sentinels when there is no reco muon / primary shower: mask the variables that need the missing object
reco_k = kin(reco["mu"], reco["g"])
for k_ in ("E_mu", "cos_mu"):
    reco_k[k_] = np.where(has_mu, reco_k[k_], np.nan)
for k_ in ("E_g", "cos_g"):
    reco_k[k_] = np.where(has_sh, reco_k[k_], np.nan)
for k_ in ("open", "M", "E_sum"):
    reco_k[k_] = np.where(has_mu & has_sh, reco_k[k_], np.nan)

In [ ]:
def reco_vs_true_1d(ax, var, bins, xlabel, stage_names=("presel + reco mu + reco shower", "numuCC + reco mu + reco shower")):
    # solid: reco; dashed: true, for the same selected events; dotted: true, all events
    ax.stairs(np.histogram(sim_k[var] * scale[var], bins, weights=cvw)[0], bins, color="gray", ls=":", label="true, all events")
    for name, c in zip(stage_names, ("C0", "C3")):
        m = stages[name]
        ax.stairs(np.histogram(reco_k[var][m] * scale[var], bins, weights=cvw[m])[0], bins, color=c, lw=1.8, label=f"reco, {name}")
        ax.stairs(np.histogram(sim_k[var][m] * scale[var], bins, weights=cvw[m])[0], bins, color=c, ls="--", label=f"true, {name}")
    ax.set(xlabel=xlabel, ylabel="events / 1e21 POT (CV)")
    ax.grid(alpha=0.3)

scale = {"E_mu": 1, "cos_mu": 1, "E_g": 1e3, "cos_g": 1, "M": 1e3, "open": 1, "E_sum": 1}
fig, axs = plt.subplots(2, 3, figsize=(17, 9))
reco_vs_true_1d(axs[0, 0], "E_mu", np.linspace(0, 2.5, 26), r"$E_\mu$ (GeV)")
reco_vs_true_1d(axs[0, 1], "cos_mu", np.linspace(-1, 1, 21), r"$\cos\theta_\mu$")
reco_vs_true_1d(axs[0, 2], "M", np.linspace(100, 1000, 37), r"$M_{\mu\gamma}$ (MeV)")
reco_vs_true_1d(axs[1, 0], "E_g", np.linspace(0, 1000, 26), r"$E_\gamma$ (MeV)")
reco_vs_true_1d(axs[1, 1], "cos_g", np.linspace(-1, 1, 21), r"$\cos\theta_\gamma$")
reco_vs_true_1d(axs[1, 2], "open", np.linspace(0, 180, 37), r"$\theta_{\mu\gamma}$ (deg)")
axs[0, 0].legend(fontsize=7)
plt.tight_layout(); save_and_show(fig, "reco_vs_true_1d")

In [ ]:
m_r = stages["presel + reco mu + reco shower"]
hist2d_pair([(reco_k["E_mu"][m_r], reco_k["cos_mu"][m_r]), (sim_k["E_mu"][m_r], sim_k["cos_mu"][m_r])],
            [(np.linspace(0, 2.5, 26), bcos)] * 2, [(r"$E_\mu$ (GeV)", r"$\cos\theta_\mu$")] * 2,
            [cvw[m_r]] * 2, ["reco", "true (same events)"], "reco_true_muon_E_vs_cos_2d", "muon energy vs angle, presel + reco mu + reco shower")
hist2d_pair([(reco_k["E_g"][m_r], reco_k["cos_g"][m_r]), (sim_k["E_g"][m_r], sim_k["cos_g"][m_r])],
            [(np.linspace(0, 1.0, 26), bcos)] * 2, [(r"$E_\gamma$ (GeV)", r"$\cos\theta_\gamma$")] * 2,
            [cvw[m_r]] * 2, ["reco", "true (same events)"], "reco_true_photon_E_vs_cos_2d", "photon energy vs angle, presel + reco mu + reco shower")

In [ ]:
# reco vs true, presel + reco mu + reco shower
pairs_rt = [("E_mu", np.linspace(0, 2.5, 26), r"$E_\mu$ (GeV)"), ("cos_mu", np.linspace(-1, 1, 21), r"$\cos\theta_\mu$"),
            ("E_g", np.linspace(0, 1000, 26), r"$E_\gamma$ (MeV)"), ("cos_g", np.linspace(-1, 1, 21), r"$\cos\theta_\gamma$"),
            ("M", np.linspace(100, 1000, 31), r"$M_{\mu\gamma}$ (MeV)"), ("open", np.linspace(0, 180, 31), r"$\theta_{\mu\gamma}$ (deg)")]
fig, axs = plt.subplots(2, 3, figsize=(16, 9))
for a, (var, b, lab) in zip(axs.flat, pairs_rt):
    H, _, _ = np.histogram2d(sim_k[var][m_r] * scale[var], reco_k[var][m_r] * scale[var], [b, b], weights=cvw[m_r])
    mm = a.pcolormesh(b, b, H.T, norm=LogNorm(vmin=max(H[H > 0].min(), H.max() * 1e-3), vmax=H.max()), cmap="viridis")
    a.plot(b, b, "r--", lw=0.8)
    plt.colorbar(mm, ax=a)
    a.set(xlabel="true " + lab, ylabel="reco " + lab)
fig.suptitle("reco vs true, presel + reco mu + reco shower (events / 1e21 POT)")
plt.tight_layout(); save_and_show(fig, "reco_vs_true_migration_2d")

### Is the photon found as its own shower?

A soft photon close to the muon is hard to reconstruct as a separate shower: it gets absorbed into the muon track, or isn't found at all. The plots
below show the CV-weighted fraction of events with a reco primary shower, among presel events with a reco muon. They are drawn as a function of the true photon energy
and of the true opening angle. A second set of curves asks whether the reco shower points along the true photon, within $20^\circ$.

In [ ]:
base = presel & has_mu
with np.errstate(invalid="ignore", divide="ignore"):
    cos_rg = np.sum(reco["g"][:, 1:] * sim["g"][:, 1:], axis=1) / (np.linalg.norm(reco["g"][:, 1:], axis=1) * np.linalg.norm(sim["g"][:, 1:], axis=1))
shower_matches_photon = has_sh & (np.degrees(np.arccos(np.clip(cos_rg, -1, 1))) < 20)

def eff_plot(ax, var, bins, xlabel, logx=False):
    for num, lab, c in [(has_sh, "reco primary shower exists", "C0"), (shower_matches_photon, "reco shower within 20 deg of true photon", "C3")]:
        d, _ = np.histogram(var[base], bins, weights=cvw[base])
        n, _ = np.histogram(var[base & num], bins, weights=cvw[base & num])
        nraw, _ = np.histogram(var[base], bins)
        with np.errstate(invalid="ignore", divide="ignore"):
            e = n / d
            err = np.sqrt(e * (1 - e) / nraw)
        xc = np.sqrt(bins[1:] * bins[:-1]) if logx else 0.5 * (bins[1:] + bins[:-1])
        ax.errorbar(xc, e, yerr=err, fmt="o-", c=c, ms=3, label=lab)
    ax.set(xlabel=xlabel, ylabel="fraction (presel + reco mu)", ylim=(0, 1))
    if logx: ax.set_xscale("log")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
eff_plot(axs[0], sim_k["E_g"] * 1e3, np.geomspace(20, 1500, 16), r"true $E_\gamma$ (MeV)", logx=True)
eff_plot(axs[1], sim_k["open"], np.linspace(0, 60, 13), r"true $\theta_{\mu\gamma}$ (deg)")
eff_plot(axs[2], sim_k["E_mu"], np.linspace(0.1, 2.5, 13), r"true $E_\mu$ (GeV)")
plt.tight_layout(); save_and_show(fig, "photon_shower_reco_fraction")

## 9. Event displays

The first 20 events in the file that have spacepoints and no primary $\pi^0$, drawn with the Wire-Cell `rec_charge_blob` spacepoints (`wcpselection/T_spacepoints`, `Trecchargeblob_spacepoints_*`).
These are the reconstructed 3D charge blobs of the in-beam (neutrino-candidate) cluster. About 60% of events have none (no neutrino candidate) and are skipped. Events with a true primary $\pi^0$ (`truth_pdg == 111`, `truth_mother == 0`) are also skipped, so the photons on display come from the radiation.
* Spacepoints are shown in the $x$-$z$ view ($z$ along the beam, horizontal; $x$ the drift coordinate, vertical), colored by charge with the jet color map. The color range is 0 to the event's 99th-percentile charge, and negative-charge points are drawn at the bottom of the scale.
* Both axes use the same scale. Each display is zoomed to the 1st-99th percentile range of the spacepoints in $x$ and in $z$, expanded if needed to include the true vertex and conversion point, plus 5 cm on each side.
* Black lines mark the TPC active-volume boundaries ($-1 < x < 254.3$ cm, $0.6 < z < 1036.4$ cm, as in `src/blip_postprocessing.py`) where they fall inside the window. They don't change the zoom.
* The magenta star is the true neutrino vertex (`T_PFeval` `truth_corr_nuvtx`). The white star is the true conversion point of the generator photon.
  Both are in the space-charge-corrected coordinates that the Wire-Cell truth uses: the `truth_startXYZT` of the primary particles equals `truth_corr_nuvtx`, not the raw `truth_vtx`.
* The conversion point is defined as in `add_extra_true_photon_variables` in `src/postprocessing.py`. Take the $e^\pm$ created directly by the generator photon
  (the Wire-Cell primary photon whose energy matches it) or by any photon descended from it, and sort them by creation time. The conversion point is the start of the
  $e^\pm$ at which their cumulative energy first passes half the photon energy. Compton electrons count towards the sum. If the sum never passes half (e.g. the photon
  leaves the TPC or its daughters were not saved), there is no conversion star.
* The box to the right lists every true primary particle (`T_PFeval` particles with `truth_mother == 0`) with its kinetic energy, highest first.
  The Wire-Cell truth keeps only final-state particles, so the list is what leaves the nucleus, including the radiated muon and the generator photon.
* The titles give the true radiated-muon and photon energies, the true opening angle, and whether the event has a reco muon and a reco primary shower after preselection.

`T_spacepoints` has no run/subrun/event branches, so it is matched to the other trees by entry number. It has the same number of entries.

In [ ]:
t_sp = f_sim["wcpselection/T_spacepoints"]
assert t_sp.num_entries == n_sim
sp = t_sp.arrays([f"Trecchargeblob_spacepoints_{c}" for c in ("x", "z", "q")])
n_sp = ak.to_numpy(ak.num(sp["Trecchargeblob_spacepoints_x"]))
print(f"events with any rec_charge_blob spacepoints: {np.mean(n_sp > 0):.3f}")

t_pf_tr = f_sim["wcpselection/T_PFeval"].arrays(["truth_id", "truth_mother", "truth_pdg", "truth_startXYZT", "truth_startMomentum",
                                                 "truth_corr_nuvtxX", "truth_corr_nuvtxY", "truth_corr_nuvtxZ"])
true_vtx = np.column_stack([ak.to_numpy(t_pf_tr[f"truth_corr_nuvtx{c_}"]) for c_ in "XYZ"]).astype(float)

def photon_conversion_point(i):
    # (x, y, z) where the generator photon's cumulative energy given to e+- (in order of creation time) first passes half its energy
    pdg_ = ak.to_numpy(t_pf_tr["truth_pdg"][i]); mo_ = ak.to_numpy(t_pf_tr["truth_mother"][i]); id_ = ak.to_numpy(t_pf_tr["truth_id"][i])
    E_ = ak.to_numpy(t_pf_tr["truth_startMomentum"][i][:, 3]); xyzt_ = ak.to_numpy(t_pf_tr["truth_startXYZT"][i])
    prim_g = np.flatnonzero((pdg_ == 22) & (mo_ == 0))
    if len(prim_g) == 0:
        return None
    g = prim_g[np.argmin(np.abs(E_[prim_g] - sim["g"][i, 0]))]
    # the photon plus any photons descended from it through photons only (usually just the photon: Geant4 keeps its track ID through a Compton scatter)
    chain = {id_[g]}
    while True:
        more = set(id_[(pdg_ == 22) & np.isin(mo_, list(chain))]) - chain
        if not more:
            break
        chain |= more
    ee = np.flatnonzero((np.abs(pdg_) == 11) & np.isin(mo_, list(chain)))
    ee = ee[np.argsort(xyzt_[ee, 3], kind="stable")]
    passed = np.flatnonzero(np.cumsum(E_[ee]) > E_[g] / 2)
    return xyzt_[ee[passed[0]], :3] if len(passed) else None

PDG_NAMES = {13: r"$\mu^-$", -13: r"$\mu^+$", 11: r"$e^-$", -11: r"$e^+$", 22: r"$\gamma$", 2212: "p", 2112: "n",
             211: r"$\pi^+$", -211: r"$\pi^-$", 111: r"$\pi^0$", 321: r"$K^+$", -321: r"$K^-$", 130: r"$K^0_L$", 310: r"$K^0_S$",
             3122: r"$\Lambda$", 3222: r"$\Sigma^+$", 3112: r"$\Sigma^-$", 3212: r"$\Sigma^0$"}

def primaries_label(i):
    # true primary particles (mother 0), kinetic energy in MeV, highest first
    pdg_ = ak.to_numpy(t_pf_tr["truth_pdg"][i]); mo_ = ak.to_numpy(t_pf_tr["truth_mother"][i])
    p4_ = ak.to_numpy(t_pf_tr["truth_startMomentum"][i])[mo_ == 0]
    pdg_ = pdg_[mo_ == 0]
    m_ = np.sqrt(np.clip(p4_[:, 3]**2 - np.sum(p4_[:, :3]**2, axis=1), 0, None))
    ke = (p4_[:, 3] - m_) * 1e3
    order = np.argsort(-ke)
    lines = [f"{PDG_NAMES.get(int(pdg_[k]), str(int(pdg_[k])))}: {ke[k]:.1f} MeV" for k in order]
    return "true primaries (KE):\n" + "\n".join(lines)

has_prim_pi0 = ak.to_numpy(ak.any((t_pf_tr["truth_pdg"] == 111) & (t_pf_tr["truth_mother"] == 0), axis=1))
print(f"events with a primary pi0: {np.mean(has_prim_pi0):.3f}; with spacepoints and no primary pi0: {np.mean((n_sp > 0) & ~has_prim_pi0):.3f}")
display_events = np.flatnonzero((n_sp > 0) & ~has_prim_pi0)[:50]
PAD_CM = 5.0
TPC_X, TPC_Z = (-1.0, 254.3), (0.6, 1036.4)   # TPC active volume (cm), as in src/blip_postprocessing.py

for i in display_events:
    x = ak.to_numpy(sp["Trecchargeblob_spacepoints_x"][i])
    z = ak.to_numpy(sp["Trecchargeblob_spacepoints_z"][i])
    q = ak.to_numpy(sp["Trecchargeblob_spacepoints_q"][i])
    conv = photon_conversion_point(i)
    stars = [pt for pt in (true_vtx[i], conv) if pt is not None]
    # 1st-99th percentile of the spacepoints, expanded to include the true vertex and conversion point, plus the padding
    x_ends = list(np.percentile(x, [1, 99]))
    z_ends = list(np.percentile(z, [1, 99]))
    x_lo, x_hi = min(x_ends + [pt[0] for pt in stars]) - PAD_CM, max(x_ends + [pt[0] for pt in stars]) + PAD_CM
    z_lo, z_hi = min(z_ends + [pt[2] for pt in stars]) - PAD_CM, max(z_ends + [pt[2] for pt in stars]) + PAD_CM

    # the figure follows the aspect of the zoom window, so equal scaling doesn't leave large empty margins
    width = 10.0
    height = float(np.clip(width * (x_hi - x_lo) / (z_hi - z_lo), 2.0, 12.0)) + 1.0
    fig, ax = plt.subplots(figsize=(width + 1.5, height))
    order = np.argsort(q)                      # draw high charge on top
    m = ax.scatter(z[order], x[order], c=q[order], cmap="jet", vmin=0, vmax=np.percentile(q, 99), s=4, lw=0)
    plt.colorbar(m, ax=ax, label="charge", fraction=0.03, pad=0.02)
    for pt, color, name in [(true_vtx[i], "magenta", r"true $\nu$ vertex"), (conv, "white", r"true $\gamma$ conversion")]:
        if pt is None:
            ax.plot([], [], "*", ms=14, mfc=color, mec="k", label=f"{name}: none found")
            continue
        label = name
        if name.endswith("conversion"):
            label += f" ({np.linalg.norm(conv - true_vtx[i]):.0f} cm from vertex)"
        ax.plot(pt[2], pt[0], "*", ms=14, mfc=color, mec="k", mew=1, label=label, zorder=5)
    ax.legend(fontsize=8, loc="best", framealpha=0.8)
    ax.text(1.13, 1.0, primaries_label(i), transform=ax.transAxes, ha="left", va="top", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec="0.6"))
    # TPC boundaries, only where they fall inside the window (they don't change the zoom)
    for xb in TPC_X:
        if x_lo < xb < x_hi:
            ax.axhline(xb, c="k", lw=1.2, zorder=1)
    for zb in TPC_Z:
        if z_lo < zb < z_hi:
            ax.axvline(zb, c="k", lw=1.2, zorder=1)
    ax.set_xlim(z_lo, z_hi); ax.set_ylim(x_lo, x_hi)
    ax.set_aspect("equal", adjustable="box")
    ax.set(xlabel="z (cm)", ylabel="x (cm)")
    ax.set_title(f"entry {i}, run {rse_vt[i, 0]} / {rse_vt[i, 1]} / {rse_vt[i, 2]}:  "
                 f"true $E_\\mu$ = {sim_k['E_mu'][i] * 1e3:.0f} MeV, $E_\\gamma$ = {sim_k['E_g'][i] * 1e3:.0f} MeV, "
                 f"$\\theta_{{\\mu\\gamma}}$ = {sim_k['open'][i]:.1f}$^\\circ$\n"
                 f"{len(x)} spacepoints; presel {bool(presel[i])}, reco mu {bool(has_mu[i])}, reco shower {bool(has_sh[i])}",
                 fontsize=9)
    plt.tight_layout(); save_and_show(fig, f"event_display_entry_{i}")


## Summary (as of 2026-10-07, seed 20261007)

**Sample structure.** All 5030 events are CC $\mu^\pm$ events (49 $\mu^+$). Each has one status-3 tree-level muon with exactly two status-1 daughters,
the radiated muon and the photon.
* Energy is conserved to $10^{-12}$ MeV, and the $\mu+\gamma$ momentum stays on the tree-muon axis.
* $|\vec p|$ drops by a median of 9 MeV. This is the documented heavier-$\mu\gamma$-system effect, and the generator gives the missing momentum to nothing.
* `T_eval`, `T_PFeval` and `nuselection` are aligned with `vertex_tree`.
* The Wire-Cell truth primary photon and muon are the generator ones (99.8% / 100%).
* Every event has $E_\gamma > 20$ MeV, $\theta_{\mu\gamma} < 60^\circ$ and $x > m_\mu/E_\mathrm{tree}$.

**Theory and paper comparisons.**
* The Python $P(E_\mathrm{tree})$ reproduces the generator algorithm exactly. Above 1 GeV the generator region ($x > m_\mu/E$) differs from the paper's Eq. 36 by under 1%. At 300 MeV it is 9% higher (0.0043 vs 0.0039), and at 150 MeV it is 1.5 times higher.
* Both Fig. 4 and Fig. 5 are reproduced in the 0.6 GeV and 2 GeV slices, in the cone dependence (cumulative) and its derivative. The emission is wide-angle for BNB muons: at 600 MeV, almost nothing falls inside $10^\circ$ and the rate is still rising at $60^\circ$. So the $60^\circ$ cut matters.
* Emission probability per POT, relative to the nominal run-5 numuCC rate: sample / theory = 1.018 ± 0.014 overall, and consistent in every $E_\mathrm{tree}$ bin. The absolute normalization of the sample (`T_pot`) is right.

**Python generator copy vs sample.**
* All true distributions agree within statistics: $E_\mu$, $\cos\theta_\mu$, $E_\gamma$, $\cos\theta_\gamma$, $M_{\mu\gamma}$, $\theta_{\mu\gamma}$, $1-x$, $E_\mathrm{tree}$, the 2D $(E_\gamma, \theta_{\mu\gamma})$ map (pull rms 1.00), and the 2D $(1-x, \eta)$ map (pull rms 0.98; the mean of 0.14 is the 1.8% normalization offset).
* The $\eta$ histogram has the worst $\chi^2$ (55/28). Its ratio is flat apart from the 1.8% overall offset, which isn't removed in the $\chi^2$.

**Reco (Wire-Cell, no BDT).**
* 39% of the sample passes the generic preselection. 15% has both a reco muon and a primary shower.
* Among presel events with a reco muon, a primary shower exists in about 25-40% of events with $E_\gamma < 60$ MeV and 65-85% above 100 MeV. However, it points within $20^\circ$ of the true photon in only about 60% of the cases where it exists. Below 60 MeV it almost never does.
* As a result, the reco $\theta_{\mu\gamma}$ has a long tail beyond the true $60^\circ$ limit, and the reco $M_{\mu\gamma}$ is broadened upward.
* The reco muon energy is biased low for the many uncontained muons.